# Updated voxel carving: synthetic visual hull

Original geometric core: Sergey Kucher and Alex Povichuk, supervised by Aaron Wetzler. Updated2026 Python translation and new demonstration maintained by Aaron Wetzler. Original source remains unchanged under `source/`. This notebook uses three **synthetic orthographic disk silhouettes**, not photographs or an original research result. The output is occupied voxels, not a marching-cubes mesh or restored iPhone app.


In [ ]:
import pathlib, sys
cwd = pathlib.Path.cwd()
UPDATED_ROOT = next((p for d in [cwd, *cwd.parents] for p in (d, d / 'updated')
                     if (p / 'volume_carving_port' / '__init__.py').exists()), None)
if UPDATED_ROOT is None:
    import tempfile, urllib.request
    UPDATED_ROOT = pathlib.Path(tempfile.mkdtemp(prefix='volume-carving-updated-'))
    package = UPDATED_ROOT / 'volume_carving_port'
    package.mkdir()
    base = 'https://raw.githubusercontent.com/twerdster/iphone-volume-carving/main/updated/'
    for name in ('__init__.py', 'sculptor.py', 'f32.py', 'vector3dex.py'):
        with urllib.request.urlopen(base + 'volume_carving_port/' + name, timeout=30) as response:
            (package / name).write_bytes(response.read())
    with urllib.request.urlopen(base + 'demo.py', timeout=30) as response:
        (UPDATED_ROOT / 'demo.py').write_bytes(response.read())
sys.path.insert(0,str(UPDATED_ROOT))
from volume_carving_port import ACCURATE, FAST, Vector3DEx
from demo import carve_demo, point_cloud_svg
from IPython.display import SVG, display
print('Updated modules:', UPDATED_ROOT)


## Edit the parameters and rerun

Use a grid size of 16,24,32 or64 and change the disk radius. More views remove voxels inconsistent with silhouettes. Three disks constrain a visual hull; they do not uniquely determine a true sphere.


In [ ]:
size, radius, mode = 24, 8, ACCURATE
model, counts = carve_demo(size, radius, mode)
print('SYNTHETIC occupied voxels before/after each view:', counts)
display(SVG(point_cloud_svg(model)))


## Compare the two recovered carving modes

Agreement on these convex synthetic masks is a regression check, not proof that all historic inputs or missing mesh code work.


In [ ]:
accurate, _ = carve_demo(16, 5, ACCURATE)
fast, _ = carve_demo(16, 5, FAST)
print('Exact occupancy agreement on this synthetic case:', accurate.bits_hex() == fast.bits_hex())
assert accurate.bits_hex() == fast.bits_hex()


## An original vector-cache bug

The compatibility vector class deliberately preserves the old cached-length behavior. Use an actual Euclidean norm rather than its stale cache in new geometry code. Carving uses explicit coordinates, not this vector-cache API.


In [ ]:
import math
v = Vector3DEx(3,0,0) + Vector3DEx(0,4,0)
print('Coordinates:', (v.x,v.y,v.z), '| original cached length:', v.length())
print('Correct Euclidean length:', math.hypot(v.x,v.y,v.z))


## Export a point cloud

This ASCII PLY export is newly added. It contains occupied voxel positions and no triangle faces.


In [ ]:
output = UPDATED_ROOT / 'build'
output.mkdir(exist_ok=True)
ply_path = output / 'notebook-synthetic-hull.ply'
ply_path.write_text(model.to_ply())
print('Wrote', ply_path, 'with', model.count_on(), 'points')


## Optional comparison against original C++

A full local checkout with a C++ compiler can compile the preserved carving component. The harness supplies declarations for missing marching-cubes types; it cannot extract a mesh. This notebook skips the comparison in a standalone/Colab download.


In [ ]:
import shutil
if (UPDATED_ROOT.parent / 'source' / 'Sculptor' / 'Sculptor.h').exists() and (shutil.which('clang++') or shutil.which('g++')):
    sys.path.insert(0,str(UPDATED_ROOT / 'harness'))
    from compare_with_original import compare
    from demo import disk_views
    print(compare(disk_views(16,5)))
else:
    print('Skipped: requires original source checkout and a local C++ compiler.')
